In [1]:
# ── CELL 1: Imports and Initialisation ──────────────────────────────────────
import ee
import geemap
import os

ee.Initialize(project='ee-festac')

# Load Amuwo Odofin boundary
amuwo_odofin = ee.FeatureCollection("FAO/GAUL/2015/level2") \
                 .filter(ee.Filter.eq('ADM0_NAME', 'Nigeria')) \
                 .filter(ee.Filter.eq('ADM1_NAME', 'Lagos')) \
                 .filter(ee.Filter.stringContains('ADM2_NAME', 'Amuwo Odofin'))

feature_count = amuwo_odofin.size().getInfo()
if feature_count == 0:
    raise ValueError("Boundary returned 0 features — check ADM2_NAME filter")

amuwo_geom = amuwo_odofin.geometry()

print("✓ GEE initialised")
print("✓ Amuwo Odofin boundary loaded and verified")
print()
print("Notebook purpose: Extract GPM IMERG 72-hour antecedent rainfall")
print("for four verified Lagos flood events as Feature 16.")

/home/deysholey/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✓ GEE initialised
✓ Amuwo Odofin boundary loaded and verified

Notebook purpose: Extract GPM IMERG 72-hour antecedent rainfall
for four verified Lagos flood events as Feature 16.


In [2]:
# ── CELL 2: Check GPM IMERG Archive Coverage ─────────────────────────────────
# Dataset: NASA/GPM_L3/IMERG_V07
# Temporal resolution: 30-minute intervals
# Spatial resolution: 0.1 degrees (~11km at Lagos latitude)
# Variable: precipitationCal — gauge-calibrated precipitation (mm/hr)
# Coverage: June 2000 to present — covers all four flood events
#
# Why 72-hour antecedent window:
# Antecedent soil moisture is the single most important determinant of
# flood response magnitude for a given rainfall event. Two identical
# 80mm rainfall events produce dramatically different inundation extents
# depending on whether the soil is already saturated from prior rain.
# 72 hours (3 days) is the standard antecedent window in flood hydrology
# literature for capturing soil moisture state in tropical urban catchments
# (Beven & Kirkby, 1979; Western et al., 2002).
#
# Unit conversion:
# IMERG provides precipitation in mm/hr at 30-minute intervals.
# To get total accumulation: sum all 30-min values × 0.5 = total mm
# (each interval represents 0.5 hours of rain)

# Verify the dataset exists and has coverage over Lagos
test = ee.ImageCollection("NASA/GPM_L3/IMERG_V07") \
         .filterBounds(amuwo_geom) \
         .filterDate('2017-07-01', '2017-07-10')

count = test.size().getInfo()
print(f"GPM IMERG archive check (Jul 2017): {count} images found")

if count == 0:
    print("⚠ Trying IMERG_V06 as fallback...")
    test_v6 = ee.ImageCollection("NASA/GPM_L3/IMERG_V06") \
                .filterBounds(amuwo_geom) \
                .filterDate('2017-07-01', '2017-07-10')
    count_v6 = test_v6.size().getInfo()
    print(f"GPM IMERG V06 fallback: {count_v6} images found")
    IMERG_ID = "NASA/GPM_L3/IMERG_V06"
else:
    IMERG_ID = "NASA/GPM_L3/IMERG_V07"

print(f"✓ Using dataset: {IMERG_ID}")
print(f"  Resolution: 0.1° (~11km) | Interval: 30 minutes")
print(f"  Variable: precipitationCal (mm/hr, gauge-calibrated)")

GPM IMERG archive check (Jul 2017): 432 images found
✓ Using dataset: NASA/GPM_L3/IMERG_V07
  Resolution: 0.1° (~11km) | Interval: 30 minutes
  Variable: precipitationCal (mm/hr, gauge-calibrated)


In [3]:
# ── CELL 3: Define GPM IMERG Extraction Function ─────────────────────────────
# For each flood event, extracts cumulative rainfall in the 72 hours
# immediately preceding the event peak date.
#
# The antecedent window is defined as:
# window_end   = midnight on the event date (flood peak day)
# window_start = 72 hours before window_end (3 days prior)
#
# This captures the rainfall that saturated the soil and drainage
# system leading up to the flood peak — not the rainfall during
# the flood itself, which is already partially captured by the
# mean annual rainfall and extreme rainfall frequency features
# from NB03 (CHIRPS). The two features are complementary:
# CHIRPS captures long-term climatological rainfall patterns.
# GPM IMERG captures event-specific antecedent forcing conditions.

def extract_imerg_72hr(event_date_start, window_label, geometry, scale=1000):
    """
    Extracts 72-hour cumulative antecedent rainfall from GPM IMERG.

    Parameters:
        event_date_start : string — flood event start date 'YYYY-MM-DD'
        window_label     : string — label for print statements
        geometry         : GEE geometry for clipping
        scale            : pixel scale in metres (default 1000m ≈ 0.01°)

    Returns:
        Single-band GEE Image of cumulative rainfall in mm
    """
    # Define 72-hour antecedent window
    event_date  = ee.Date(event_date_start)
    window_end  = event_date
    window_start= event_date.advance(-3, 'day')

    print(f"\n{window_label}")
    print(f"  Antecedent window: {window_start.format('YYYY-MM-dd').getInfo()} "
          f"→ {window_end.format('YYYY-MM-dd').getInfo()}")

    # Load IMERG collection for the 72-hour window
    imerg = ee.ImageCollection(IMERG_ID) \
              .filterBounds(geometry) \
              .filterDate(window_start, window_end) \
              .select('precipitation')

    image_count = imerg.size().getInfo()
    print(f"  IMERG images in window: {image_count}")

    if image_count == 0:
        print("  ⚠ No IMERG data found in this window")
        return None

    # Sum all 30-minute precipitation values
    # Each image = mm/hr rate for a 30-minute period
    # Multiply sum by 0.5 to convert from mm/hr to mm accumulated
    # (sum of rates × 0.5hr per interval = total mm)
    cumulative_rainfall = imerg.sum() \
                               .multiply(0.5) \
                               .clip(geometry) \
                               .rename('imerg_72hr_rainfall')

    # Verify statistics
    stats = cumulative_rainfall.reduceRegion(
        reducer   = ee.Reducer.minMax().combine(
            reducer2  = ee.Reducer.mean(),
            sharedInputs = True
        ),
        geometry  = geometry,
        scale     = scale,
        maxPixels = 1e9
    ).getInfo()

    r_min  = stats.get('imerg_72hr_rainfall_min', 0)
    r_max  = stats.get('imerg_72hr_rainfall_max', 0)
    r_mean = stats.get('imerg_72hr_rainfall_mean', 0)

    print(f"  ✓ 72-hr cumulative rainfall computed")
    print(f"  Minimum : {r_min:.2f} mm")
    print(f"  Maximum : {r_max:.2f} mm")
    print(f"  Mean    : {r_mean:.2f} mm")

    # Quality check — verify against documented rainfall magnitudes
    # Event 1 (Jul 2017): 176.5mm documented by NiMet
    # Event 2 (Jun 2020): ~90mm documented by FloodList/LASEMA
    # Events 3 and 4 should show similarly elevated antecedent totals
    if r_mean < 5:
        print("  ⚠ WARNING: Mean < 5mm — check window dates and IMERG coverage")
    elif r_mean > 300:
        print("  ⚠ WARNING: Mean > 300mm — check unit conversion (× 0.5 applied?)")
    else:
        print("  ✓ Rainfall magnitude within plausible range for Lagos rainy season")

    return cumulative_rainfall

print("✓ GPM IMERG extraction function defined")
print("  Unit: mm cumulative over 72-hour antecedent window")
print("  Conversion: sum(mm/hr) × 0.5hr = total mm accumulated")

✓ GPM IMERG extraction function defined
  Unit: mm cumulative over 72-hour antecedent window
  Conversion: sum(mm/hr) × 0.5hr = total mm accumulated


In [4]:
# ── CELL 4: Extract 72-Hour Antecedent Rainfall for All Four Events ──────────
# Antecedent windows set to 72 hours before each confirmed event date.
# Event dates confirmed through independent re-verification in NB05 planning:
# Event 1: 07-08 July 2017  → window: 04-07 July 2017
# Event 2: 17-19 June 2020  → window: 14-17 June 2020
# Event 3: 16 July 2021     → window: 13-16 July 2021
# Event 4: 09-10 July 2022  → window: 06-09 July 2022

print("=" * 55)
print("  GPM IMERG 72-HOUR RAINFALL EXTRACTION")
print("=" * 55)

rainfall_maps = {}

# Event 1: 07-08 July 2017
rainfall_maps['rain_2017'] = extract_imerg_72hr(
    event_date_start = '2017-07-07',
    window_label     = 'Event 1: 07-08 July 2017',
    geometry         = amuwo_geom
)

# Event 2: 17-19 June 2020
rainfall_maps['rain_2020'] = extract_imerg_72hr(
    event_date_start = '2020-06-17',
    window_label     = 'Event 2: 17-19 June 2020',
    geometry         = amuwo_geom
)

# Event 3: 16 July 2021
rainfall_maps['rain_2021'] = extract_imerg_72hr(
    event_date_start = '2021-07-16',
    window_label     = 'Event 3: 16 July 2021',
    geometry         = amuwo_geom
)

# Event 4: 09-10 July 2022
rainfall_maps['rain_2022'] = extract_imerg_72hr(
    event_date_start = '2022-07-09',
    window_label     = 'Event 4: 09-10 July 2022',
    geometry         = amuwo_geom
)

# Summary
print()
print("=" * 55)
successful = [k for k, v in rainfall_maps.items() if v is not None]
print(f"  Successful extractions: {len(successful)} of 4")
for e in successful:
    print(f"  ✓ {e}")

  GPM IMERG 72-HOUR RAINFALL EXTRACTION

Event 1: 07-08 July 2017
  Antecedent window: 2017-07-04 → 2017-07-07
  IMERG images in window: 144
  ✓ 72-hr cumulative rainfall computed
  Minimum : 110.14 mm
  Maximum : 134.29 mm
  Mean    : 126.20 mm
  ✓ Rainfall magnitude within plausible range for Lagos rainy season

Event 2: 17-19 June 2020
  Antecedent window: 2020-06-14 → 2020-06-17
  IMERG images in window: 144
  ✓ 72-hr cumulative rainfall computed
  Minimum : 25.94 mm
  Maximum : 40.65 mm
  Mean    : 34.01 mm
  ✓ Rainfall magnitude within plausible range for Lagos rainy season

Event 3: 16 July 2021
  Antecedent window: 2021-07-13 → 2021-07-16
  IMERG images in window: 144
  ✓ 72-hr cumulative rainfall computed
  Minimum : 3.69 mm
  Maximum : 11.92 mm
  Mean    : 10.09 mm
  ✓ Rainfall magnitude within plausible range for Lagos rainy season

Event 4: 09-10 July 2022
  Antecedent window: 2022-07-06 → 2022-07-09
  IMERG images in window: 144
  ✓ 72-hr cumulative rainfall computed
  Min

In [5]:
# ── CELL 5: Mean Antecedent Rainfall Composite — Feature 16 ──────────────────
# The mean across all four events represents the typical antecedent
# rainfall forcing associated with flood-producing conditions in
# Amuwo Odofin. This composite becomes Feature 16 in the ML feature
# matrix — a single layer capturing event-specific rainfall forcing
# as a conditioning factor alongside the static terrain features.
#
# Why a composite rather than four separate features?
# Four separate event rainfall layers would introduce event-specific
# temporal variation into a spatial susceptibility model, creating
# collinearity with the flood label itself (the model would simply
# learn which event produced more rain rather than which terrain
# is inherently susceptible). The mean composite captures the
# spatial pattern of rainfall distribution across Amuwo Odofin
# — which parts consistently receive higher rainfall during major
# events due to mesoscale rainfall gradients — without encoding
# event-specific temporal variation.

valid_maps = [v for v in rainfall_maps.values() if v is not None]

if len(valid_maps) == 0:
    raise ValueError("No valid rainfall maps — check IMERG availability")

# Mean composite across all valid events
mean_antecedent_rainfall = ee.ImageCollection(valid_maps) \
                             .mean() \
                             .rename('gpm_antecedent_rainfall') \
                             .toFloat()

# Final statistics for methods log
composite_stats = mean_antecedent_rainfall.reduceRegion(
    reducer   = ee.Reducer.minMax().combine(
        reducer2  = ee.Reducer.mean().combine(
            reducer2  = ee.Reducer.stdDev(),
            sharedInputs = True
        ),
        sharedInputs = True
    ),
    geometry  = amuwo_geom,
    scale     = 1000,
    maxPixels = 1e9
).getInfo()

print("✓ Mean antecedent rainfall composite computed (Feature 16)")
print()
print("GPM IMERG Composite Statistics (mm/72hr):")
print(f"  Minimum  : {composite_stats.get('gpm_antecedent_rainfall_min', 0):.2f} mm")
print(f"  Maximum  : {composite_stats.get('gpm_antecedent_rainfall_max', 0):.2f} mm")
print(f"  Mean     : {composite_stats.get('gpm_antecedent_rainfall_mean', 0):.2f} mm")
print(f"  Std Dev  : {composite_stats.get('gpm_antecedent_rainfall_stdDev', 0):.2f} mm")

✓ Mean antecedent rainfall composite computed (Feature 16)

GPM IMERG Composite Statistics (mm/72hr):
  Minimum  : 48.09 mm
  Maximum  : 62.37 mm
  Mean     : 57.25 mm
  Std Dev  : 3.77 mm


In [6]:
# ── CELL 6: Visualisation ────────────────────────────────────────────────────

amuwo_centroid = amuwo_odofin.geometry().centroid().coordinates().getInfo()
lon, lat = amuwo_centroid[0], amuwo_centroid[1]

Map = geemap.Map(center=[lat, lon], zoom=11)

amuwo_style = {'color': 'FF0000', 'fillColor': '00000000', 'width': 3}
Map.addLayer(amuwo_odofin.style(**amuwo_style), {}, 'Amuwo Odofin Boundary')

# Individual event rainfall maps
event_colours = {
    'rain_2017': ['FFFFCC', '41B6C4', '253494'],
    'rain_2020': ['FFFFCC', '78C679', '006837'],
    'rain_2021': ['FFFFCC', 'FEB24C', 'BD0026'],
    'rain_2022': ['FFFFCC', 'A50F15', '67000D']
}

event_labels = {
    'rain_2017': 'Antecedent Rainfall - Jul 2017 (mm)',
    'rain_2020': 'Antecedent Rainfall - Jun 2020 (mm)',
    'rain_2021': 'Antecedent Rainfall - Jul 2021 (mm)',
    'rain_2022': 'Antecedent Rainfall - Jul 2022 (mm)'
}

for key, img in rainfall_maps.items():
    if img is not None:
        Map.addLayer(img, {
            'min': 0, 'max': 200,
            'palette': event_colours[key]
        }, event_labels[key])

# Mean composite — primary feature layer
Map.addLayer(mean_antecedent_rainfall, {
    'min': 0, 'max': 150,
    'palette': ['FFFFFF', 'ADD8E6', '0000FF', '000033']
}, 'Mean Antecedent Rainfall Composite — Feature 16 (mm)')

Map.addLayerControl()
Map

Map(center=[6.455061708555176, 3.265078291524212], controls=(WidgetControl(options=['position', 'transparent_b…

In [9]:
# ── CELL 7: Export to Google Drive ───────────────────────────────────────────
# Export 1: Mean composite — this becomes Feature 16 in the ML matrix
# Export 2-5: Individual event rainfall maps — used in NB05 revised
# flood inventory for event-specific antecedent rainfall labelling

# ── Export mean composite (Feature 16) ───────────────────────────────────────
task_composite = ee.batch.Export.image.toDrive(
    image          = mean_antecedent_rainfall,
    description    = 'amuwo_odofin_gpm_antecedent_rainfall',
    folder         = 'GeoAID_Project',
    fileNamePrefix = 'gpm_antecedent_rainfall_amuwo_odofin',
    region         = amuwo_geom,
    scale          = 500,
    crs            = 'EPSG:4326',
    maxPixels      = 1e9,
    fileFormat     = 'GeoTIFF'
)
task_composite.start()
print("✓ Export 1: gpm_antecedent_rainfall_amuwo_odofin.tif (Feature 16)")

# ── Export individual event maps ──────────────────────────────────────────────
event_exports = {
    'rain_2017': ('gpm_rainfall_2017', 'GPM 72hr rainfall Jul 2017'),
    'rain_2020': ('gpm_rainfall_2020', 'GPM 72hr rainfall Jun 2020'),
    'rain_2021': ('gpm_rainfall_2021', 'GPM 72hr rainfall Jul 2021'),
    'rain_2022': ('gpm_rainfall_2022', 'GPM 72hr rainfall Jul 2022')
}

for key, (prefix, description) in event_exports.items():
    if rainfall_maps.get(key) is not None:
        task = ee.batch.Export.image.toDrive(
            image          = rainfall_maps[key],
            description    = f'amuwo_odofin_{prefix}',
            folder         = 'GeoAID_Project',
            fileNamePrefix = f'{prefix}_amuwo_odofin',
            region         = amuwo_geom,
            scale          = 1000,
            crs            = 'EPSG:4326',
            maxPixels      = 1e9,
            fileFormat     = 'GeoTIFF'
        )
        task.start()
        print(f"✓ Export: {prefix}_amuwo_odofin.tif ({description})")

print()
print("Monitor at: https://code.earthengine.google.com/tasks")

✓ Export 1: gpm_antecedent_rainfall_amuwo_odofin.tif (Feature 16)
✓ Export: gpm_rainfall_2017_amuwo_odofin.tif (GPM 72hr rainfall Jul 2017)
✓ Export: gpm_rainfall_2020_amuwo_odofin.tif (GPM 72hr rainfall Jun 2020)
✓ Export: gpm_rainfall_2021_amuwo_odofin.tif (GPM 72hr rainfall Jul 2021)
✓ Export: gpm_rainfall_2022_amuwo_odofin.tif (GPM 72hr rainfall Jul 2022)

Monitor at: https://code.earthengine.google.com/tasks


In [8]:
# ── CELL 8: Session Summary ──────────────────────────────────────────────────
print("=" * 60)
print("  NOTEBOOK 04b — GPM IMERG ANTECEDENT RAINFALL: COMPLETE")
print("=" * 60)
print()
print("  Dataset: NASA/GPM_L3/IMERG_V07 (gauge-calibrated)")
print("  Variable: precipitationCal (mm/hr × 0.5hr = mm accumulated)")
print("  Antecedent window: 72 hours before each event peak date")
print()
print("  Events processed:")
print("  ├── 72hr before 07 Jul 2017 (04-07 Jul 2017)")
print("  ├── 72hr before 17 Jun 2020 (14-17 Jun 2020)")
print("  ├── 72hr before 16 Jul 2021 (13-16 Jul 2021)")
print("  └── 72hr before 09 Jul 2022 (06-09 Jul 2022)")
print()
print("  Output:")
print("  ├── gpm_antecedent_rainfall_amuwo_odofin.tif → Feature 16 ✓")
print("  ├── gpm_rainfall_2017_amuwo_odofin.tif")
print("  ├── gpm_rainfall_2020_amuwo_odofin.tif")
print("  ├── gpm_rainfall_2021_amuwo_odofin.tif")
print("  └── gpm_rainfall_2022_amuwo_odofin.tif")
print()
print("  DATA ACQUISITION PHASE: FULLY COMPLETE")
print("  All 16 features acquired across 5 GeoTIFF files:")
print("  ├── topo_features_amuwo_odofin.tif (F1-F6)")
print("  ├── rainfall_lulc_ndvi_amuwo_odofin.tif (F7-F11)")
print("  ├── soil_distance_amuwo_odofin.tif (F12-F14)")
print("  └── gpm_antecedent_rainfall_amuwo_odofin.tif (F15)")
print()
print("  Next: 05_flood_inventory_SAR.ipynb (REBUILD)")
print("        → Corrected water body masking")
print("        → JRC threshold tightened to 5% occurrence")
print("        → ESA WorldCover urban land mask added")
print("        → Minimum backscatter change increased to 3.0dB")
print("=" * 60)

  NOTEBOOK 04b — GPM IMERG ANTECEDENT RAINFALL: COMPLETE

  Dataset: NASA/GPM_L3/IMERG_V07 (gauge-calibrated)
  Variable: precipitationCal (mm/hr × 0.5hr = mm accumulated)
  Antecedent window: 72 hours before each event peak date

  Events processed:
  ├── 72hr before 07 Jul 2017 (04-07 Jul 2017)
  ├── 72hr before 17 Jun 2020 (14-17 Jun 2020)
  ├── 72hr before 16 Jul 2021 (13-16 Jul 2021)
  └── 72hr before 09 Jul 2022 (06-09 Jul 2022)

  Output:
  ├── gpm_antecedent_rainfall_amuwo_odofin.tif → Feature 16 ✓
  ├── gpm_rainfall_2017_amuwo_odofin.tif
  ├── gpm_rainfall_2020_amuwo_odofin.tif
  ├── gpm_rainfall_2021_amuwo_odofin.tif
  └── gpm_rainfall_2022_amuwo_odofin.tif

  DATA ACQUISITION PHASE: FULLY COMPLETE
  All 16 features acquired across 5 GeoTIFF files:
  ├── topo_features_amuwo_odofin.tif (F1-F6)
  ├── rainfall_lulc_ndvi_amuwo_odofin.tif (F7-F11)
  ├── soil_distance_amuwo_odofin.tif (F12-F14)
  └── gpm_antecedent_rainfall_amuwo_odofin.tif (F15)

  Next: 05_flood_inventory_SAR.ipy

In [10]:
# ── DIAGNOSTIC: Inspect GPM composite before export ──────────────────────────
import ee
ee.Initialize(project='ee-festac')

# Reload boundary
amuwo_odofin = ee.FeatureCollection("FAO/GAUL/2015/level2") \
                 .filter(ee.Filter.eq('ADM0_NAME', 'Nigeria')) \
                 .filter(ee.Filter.eq('ADM1_NAME', 'Lagos')) \
                 .filter(ee.Filter.stringContains('ADM2_NAME', 'Amuwo Odofin'))

amuwo_geom = amuwo_odofin.geometry()

# Check geometry bounds
bounds = amuwo_geom.bounds().getInfo()
print("Study area bounds:")
print(bounds)
print()

# Check the mean composite has valid pixels
# Rebuild directly here rather than relying on session variables
IMERG_ID = "NASA/GPM_L3/IMERG_V07"

events = [
    ('2017-07-04', '2017-07-07'),
    ('2020-06-14', '2020-06-17'),
    ('2021-07-13', '2021-07-16'),
    ('2022-07-06', '2022-07-09'),
]

rainfall_images = []
for start, end in events:
    img = ee.ImageCollection(IMERG_ID) \
             .filterBounds(amuwo_geom) \
             .filterDate(start, end) \
             .select('precipitation') \
             .sum() \
             .multiply(0.5) \
             .clip(amuwo_geom)
    rainfall_images.append(img)

mean_composite = ee.ImageCollection(rainfall_images) \
                   .mean() \
                   .rename('gpm_antecedent_rainfall') \
                   .toFloat()

# Check pixel count and stats
stats = mean_composite.reduceRegion(
    reducer   = ee.Reducer.minMax().combine(
        reducer2  = ee.Reducer.count(),
        sharedInputs = True
    ),
    geometry  = amuwo_geom,
    scale     = 500,
    maxPixels = 1e9
).getInfo()

print("GPM composite statistics at 500m scale:")
for k, v in stats.items():
    print(f"  {k}: {v}")
print()

# Submit corrected export
task = ee.batch.Export.image.toDrive(
    image          = mean_composite,
    description    = 'amuwo_odofin_gpm_antecedent_rainfall_v2',
    folder         = 'GeoAID_Project',
    fileNamePrefix = 'gpm_antecedent_rainfall_amuwo_odofin',
    region         = amuwo_geom,
    scale          = 500,
    crs            = 'EPSG:4326',
    maxPixels      = 1e9,
    fileFormat     = 'GeoTIFF'
)
task.start()
print("✓ Export submitted")
print("Monitor at: https://code.earthengine.google.com/tasks")

Study area bounds:
{'geodesic': False, 'type': 'Polygon', 'coordinates': [[[3.1845874162762797, 6.397226997910984], [3.369292280920736, 6.397226997910984], [3.369292280920736, 6.528659034345632], [3.1845874162762797, 6.528659034345632], [3.1845874162762797, 6.397226997910984]]]}

GPM composite statistics at 500m scale:
  gpm_antecedent_rainfall_count: 711
  gpm_antecedent_rainfall_max: 64.8499984741211
  gpm_antecedent_rainfall_min: 48.09499740600586

✓ Export submitted
Monitor at: https://code.earthengine.google.com/tasks
